# Notebook 04 — Model B: ClinicalBERT Transformer
**Week 3**: Fine-tune Bio_ClinicalBERT for multi-label ICD-10 prediction with section-based truncation for long discharge notes.

In [ ]:
# Install PyTorch with CUDA 12.4 (run ONCE in a regular terminal, not here):
# python -m pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
#
# Then install remaining deps:
# pip install -q transformers datasets accelerate pandas pyarrow numpy scikit-learn

In [ ]:
# Running locally — no Drive mount needed

## 1. Config

In [ ]:
import os, json, pickle, re
import numpy as np
import pandas as pd
import torch
from pathlib import Path

DATA_DIR  = '../datasets/processed'
MODEL_DIR = '../data/models/model_b'
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_NAME    = 'emilyalsentzer/Bio_ClinicalBERT'
MAX_SEQ_LEN   = 512
LR            = 2e-5
EPOCHS        = 3
WARMUP_RATIO  = 0.1
SEED          = 42
USE_AMP       = True   # mixed precision (fp16) — ~2x faster on CUDA

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Auto-detect GPU and set optimal batch size
# Effective batch size is always 32 (BATCH_SIZE × GRAD_ACCUM)
if torch.cuda.is_available():
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9
    if gpu_mem_gb >= 70:        # A100 80GB
        BATCH_SIZE = 128; GRAD_ACCUM = 1
    elif gpu_mem_gb >= 38:      # A100 40GB
        BATCH_SIZE = 64;  GRAD_ACCUM = 1
    elif gpu_mem_gb >= 22:      # L4 24GB
        BATCH_SIZE = 32;  GRAD_ACCUM = 1
    elif gpu_mem_gb >= 14:      # T4 / V100 16GB
        BATCH_SIZE = 16;  GRAD_ACCUM = 2
    else:                       # smaller (GTX 1650 etc.)
        BATCH_SIZE = 4;   GRAD_ACCUM = 8
    print(f'✓ GPU DETECTED: {gpu_name} ({gpu_mem_gb:.0f} GB VRAM)')
    print(f'✓ CUDA version: {torch.version.cuda}')
    print(f'✓ Mixed precision (fp16): ENABLED')
else:
    BATCH_SIZE = 4; GRAD_ACCUM = 8
    print('⚠ WARNING: No GPU detected — training will be VERY slow on CPU!')
    print('  Make sure CUDA-enabled PyTorch is installed:')
    print('  pip install torch --index-url https://download.pytorch.org/whl/cu124')

print(f'Device: {device}  |  Batch: {BATCH_SIZE}  GradAccum: {GRAD_ACCUM}  →  Effective: {BATCH_SIZE * GRAD_ACCUM}')

## 2. Load data & label binarizer

In [ ]:
train_df = pd.read_parquet(f'{DATA_DIR}/cohort_train_clean.parquet')
val_df   = pd.read_parquet(f'{DATA_DIR}/cohort_val_clean.parquet')
test_df  = pd.read_parquet(f'{DATA_DIR}/cohort_test_clean.parquet')

with open(f'{DATA_DIR}/mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
vocab = list(mlb.classes_)
NUM_LABELS = len(vocab)

for df_ in [train_df, val_df, test_df]:
    df_['icd_codes'] = df_['icd_codes_str'].str.split('|')

Y_train = mlb.transform(train_df['icd_codes']).astype(np.float32)
Y_val   = mlb.transform(val_df['icd_codes']).astype(np.float32)
Y_test  = mlb.transform(test_df['icd_codes']).astype(np.float32)

print(f'Labels: {NUM_LABELS}  Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

## 3. Section-based text truncation (long-text strategy)
Priority order: Discharge Diagnoses → Hospital Course → Chief Complaint → beginning of note

In [ ]:
# Section priority for clinical relevance
_SECTION_PATTERNS = [
    r'discharge diagnos[ei]s.*?(?=\n[A-Z][A-Z ]{3,}:|$)',
    r'discharge condition.*?(?=\n[A-Z][A-Z ]{3,}:|$)',
    r'hospital course.*?(?=\n[A-Z][A-Z ]{3,}:|$)',
    r'history of present illness.*?(?=\n[A-Z][A-Z ]{3,}:|$)',
    r'chief complaint.*?(?=\n[A-Z][A-Z ]{3,}:|$)',
]
_COMPILED = [re.compile(p, re.IGNORECASE | re.DOTALL) for p in _SECTION_PATTERNS]

def smart_truncate(text: str, max_chars: int = 2000) -> str:
    """Extract high-priority sections first, then fill up to max_chars."""
    segments = []
    total = 0
    for pat in _COMPILED:
        m = pat.search(text)
        if m:
            seg = m.group(0).strip()
            if total + len(seg) <= max_chars:
                segments.append(seg)
                total += len(seg)
    if total < max_chars:
        # Fill remainder from beginning of note
        segments.insert(0, text[:max_chars - total])
    return ' '.join(segments)[:max_chars]

# max_chars ≈ MAX_SEQ_LEN * ~4 chars/token
CHAR_LIMIT = MAX_SEQ_LEN * 4
for df_ in [train_df, val_df, test_df]:
    df_['model_text'] = df_['clean_text'].apply(lambda t: smart_truncate(t, CHAR_LIMIT))

print('Sample truncated text (first 400 chars):')
print(train_df['model_text'].iloc[0][:400])

## 4. Dataset & DataLoader

In [ ]:
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ICDDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts  = list(texts)
        self.labels = labels   # np.ndarray (n, num_labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=MAX_SEQ_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_ds = ICDDataset(train_df['model_text'], Y_train)
val_ds   = ICDDataset(val_df['model_text'],   Y_val)
test_ds  = ICDDataset(test_df['model_text'],  Y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 5. Model — BERT with sigmoid multi-label head

In [ ]:
from transformers import AutoModel
import torch.nn as nn

class ICDClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.bert    = AutoModel.from_pretrained(model_name)
        hidden_size  = self.bert.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]   # [CLS] token
        cls = self.dropout(cls)
        return self.head(cls)   # raw logits

model = ICDClassifier(MODEL_NAME, NUM_LABELS).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 6. Training loop

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score

total_steps   = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.BCEWithLogitsLoss()
scaler    = torch.amp.GradScaler(device, enabled=USE_AMP and device == 'cuda')

def val_micro_f1(loader, threshold=0.5):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            with torch.amp.autocast(device, enabled=USE_AMP and device == 'cuda'):
                logits = model(ids, mask)
            probs  = torch.sigmoid(logits).cpu().float().numpy()
            all_preds.append(probs)
            all_labels.append(batch['labels'].numpy())
    P = np.vstack(all_preds)
    Y = np.vstack(all_labels)
    return f1_score(Y, (P >= threshold).astype(int), average='micro', zero_division=0), P, Y

best_val_f1   = 0.0
history       = []

# ── GPU utilization check before training ──
if device == 'cuda':
    alloc = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'✓ Model loaded on GPU  |  VRAM used: {alloc:.1f} GB / {total:.0f} GB')
    print(f'✓ fp16 mixed precision: {"ON" if USE_AMP else "OFF"}')
    print(f'✓ Training: {EPOCHS} epochs × {len(train_loader)} batches (batch_size={BATCH_SIZE})')
    print(f'✓ Total optimizer steps: {total_steps}')
    print()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.amp.autocast(device, enabled=USE_AMP and device == 'cuda'):
            logits = model(ids, mask)
            loss   = criterion(logits, labels) / GRAD_ACCUM

        scaler.scale(loss).backward()
        total_loss += loss.item() * GRAD_ACCUM

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        # Progress logging
        if (step + 1) % 500 == 0:
            if device == 'cuda':
                gpu_mem = torch.cuda.memory_allocated() / 1e9
                print(f'  Step {step+1}/{len(train_loader)}  loss={total_loss/(step+1):.4f}  GPU mem={gpu_mem:.1f}GB')
            else:
                print(f'  Step {step+1}/{len(train_loader)}  loss={total_loss/(step+1):.4f}')

        # Mid-epoch checkpoint (safety against disconnects/crashes)
        if (step + 1) % 2000 == 0:
            torch.save({
                'epoch': epoch, 'step': step,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict(),
            }, f'{MODEL_DIR}/checkpoint_e{epoch}_s{step+1}.pt')
            print(f'  → mid-epoch checkpoint saved')

    avg_loss = total_loss / len(train_loader)
    val_f1, P_val, Y_val_np = val_micro_f1(val_loader)
    history.append({'epoch': epoch, 'train_loss': avg_loss, 'val_micro_f1': val_f1})
    print(f'Epoch {epoch}/{EPOCHS}  loss={avg_loss:.4f}  val_micro_F1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f'{MODEL_DIR}/best_model.pt')
        np.save(f'{MODEL_DIR}/P_val_best.npy',  P_val)
        np.save(f'{MODEL_DIR}/Y_val_best.npy',  Y_val_np)
        print('  → saved best checkpoint')

pd.DataFrame(history).to_csv(f'{MODEL_DIR}/training_history.csv', index=False)
print('Training complete. Best val micro-F1:', best_val_f1)

## 7. Threshold tuning & test evaluation

In [ ]:
# Reload best checkpoint
model.load_state_dict(torch.load(f'{MODEL_DIR}/best_model.pt', map_location=device))

# Tune threshold on val
P_val_best = np.load(f'{MODEL_DIR}/P_val_best.npy')
Y_val_best = np.load(f'{MODEL_DIR}/Y_val_best.npy')

best_t, best_vf1 = 0.5, 0.0
for t in np.arange(0.05, 0.65, 0.025):
    vf1 = f1_score(Y_val_best, (P_val_best >= t).astype(int), average='micro', zero_division=0)
    if vf1 > best_vf1:
        best_vf1, best_t = vf1, t
print(f'Best threshold: {best_t:.3f}  val micro-F1: {best_vf1:.4f}')

# Test evaluation
from sklearn.metrics import precision_score, recall_score, average_precision_score, roc_auc_score

test_f1, P_test, Y_test_np = val_micro_f1(test_loader, threshold=best_t)
preds_test = (P_test >= best_t).astype(int)

mask = Y_test_np.sum(0) > 0
results = {
    'threshold'   : best_t,
    'micro_f1'    : f1_score(Y_test_np, preds_test, average='micro',  zero_division=0),
    'macro_f1'    : f1_score(Y_test_np, preds_test, average='macro',  zero_division=0),
    'micro_prec'  : precision_score(Y_test_np, preds_test, average='micro', zero_division=0),
    'micro_rec'   : recall_score(Y_test_np, preds_test, average='micro',    zero_division=0),
    'macro_auprc' : average_precision_score(Y_test_np[:, mask], P_test[:, mask], average='macro'),
    'micro_auroc' : roc_auc_score(Y_test_np[:, mask], P_test[:, mask], average='micro'),
}
for k, v in results.items():
    print(f'  {k:20s}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

with open(f'{MODEL_DIR}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

np.save(f'{MODEL_DIR}/P_test.npy', P_test)
np.save(f'{MODEL_DIR}/Y_test.npy', Y_test_np)
print('Test results & probabilities saved.')

## 8. Head vs. tail analysis

In [ ]:
import matplotlib.pyplot as plt

Y_train_full = np.load(f'{DATA_DIR}/Y_train.npy')
per_label_freq = Y_train_full.sum(0)
per_label_f1   = f1_score(Y_test_np, preds_test, average=None, zero_division=0)

label_df = pd.DataFrame({'icd_code': vocab, 'train_freq': per_label_freq, 'test_f1': per_label_f1})

for lo, hi, name in [(500, 1e9, 'head (≥500)'), (100, 499, 'torso (100-499)'), (0, 99, 'tail (<100)')]:
    s = label_df[(label_df['train_freq'] >= lo) & (label_df['train_freq'] <= hi)]
    print(f'{name:25s}  n={len(s):5d}  avg_F1={s["test_f1"].mean():.4f}')

plt.figure(figsize=(7, 4))
plt.scatter(label_df['train_freq'].clip(upper=2000), label_df['test_f1'], alpha=0.3, s=8)
plt.xscale('log'); plt.xlabel('Train freq (log)'); plt.ylabel('Test F1')
plt.title('Per-label F1 vs training frequency (Model B)')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/head_tail_f1.png', dpi=120); plt.show()